第一节课，有关模型的调用和输出格式的规范化

In [1]:
import os
import openai
import requests
from openai import OpenAI

from dotenv import load_dotenv
load_dotenv()
openai.api_key = os.getenv("OPENAI_API_KEY")

In [2]:
def get_completion(prompt, model="LongCat-Flash-Chat"):
    client = OpenAI(
        api_key=os.getenv("OPENAI_API_KEY"),
        base_url="https://api.longcat.chat/openai"
    )
    messages = [{"role": "user", "content": prompt}]
    response = client.chat.completions.create(
        model="LongCat-Flash-Chat",
        messages=messages,
        max_tokens=1000,
        temperature=0
    )
    return response.choices[0].message.content

In [3]:
get_completion("What is 1 + 1?")

"1 + 1 equals 2.\n\nThis is a fundamental arithmetic fact in standard base-10 mathematics. If you're exploring different contexts (like binary, modular arithmetic, or abstract algebra), the result might vary—but under normal assumptions, **1 + 1 = 2**. 😊"

In [4]:
from langchain_openai import ChatOpenAI
chat = ChatOpenAI(api_key=os.getenv("OPENAI_API_KEY"),
                  base_url="https://api.longcat.chat/openai",
                  model="LongCat-Flash-Chat",
                  max_tokens=1000,
                  temperature=0)
chat

ChatOpenAI(profile={}, client=<openai.resources.chat.completions.completions.Completions object at 0x00000173A04053F0>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x00000173A0D5D180>, root_client=<openai.OpenAI object at 0x00000173A04042E0>, root_async_client=<openai.AsyncOpenAI object at 0x00000173A0D5C4C0>, model_name='LongCat-Flash-Chat', temperature=0.0, model_kwargs={}, openai_api_key=SecretStr('**********'), openai_api_base='https://api.longcat.chat/openai', max_tokens=1000)

In [5]:
from langchain_core.prompts import ChatPromptTemplate


In [6]:
template_string = """Translate the text that \
is delimited by triple backticks \
into a style that is {style}.\
Text:```{text}```
"""

In [7]:
prompt = ChatPromptTemplate.from_template(template_string)

In [8]:
prompt.messages[0].prompt

PromptTemplate(input_variables=['style', 'text'], input_types={}, partial_variables={}, template='Translate the text that is delimited by triple backticks into a style that is {style}.Text:```{text}```\n')

In [9]:
prompt.messages[0].input_variables

['style', 'text']

In [10]:
prompt.messages

[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['style', 'text'], input_types={}, partial_variables={}, template='Translate the text that is delimited by triple backticks into a style that is {style}.Text:```{text}```\n'), additional_kwargs={})]

In [11]:
custom_style = "American English in a calm and friendly tone"

In [12]:
custom_text = "I'm going to the store to buy some groceries."

In [13]:
custom_message = prompt.format_messages(style=custom_style, text=custom_text)

In [14]:
print(custom_message)

[HumanMessage(content="Translate the text that is delimited by triple backticks into a style that is American English in a calm and friendly tone.Text:```I'm going to the store to buy some groceries.```\n", additional_kwargs={}, response_metadata={})]


In [15]:
print(type(custom_message))

<class 'list'>


In [16]:
print(type(custom_message[0]))

<class 'langchain_core.messages.human.HumanMessage'>


In [17]:
print(custom_message[0])

content="Translate the text that is delimited by triple backticks into a style that is American English in a calm and friendly tone.Text:```I'm going to the store to buy some groceries.```\n" additional_kwargs={} response_metadata={}


In [18]:
custom_response = chat.invoke(custom_message)

In [19]:
print(custom_response)

content="I'm heading to the store to pick up a few groceries." additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 14, 'prompt_tokens': 51, 'total_tokens': 65, 'completion_tokens_details': None, 'prompt_tokens_details': None, 'cache_write_tokens': 0, 'cache_read_tokens': 0, 'input_tokens': 0, 'output_tokens': 0, 'cached_tokens': 0}, 'model_provider': 'openai', 'model_name': 'longcat-flash-chatai-api', 'system_fingerprint': None, 'id': '3c40500f375e47c88c8e3784b6fd39f0', 'finish_reason': 'stop', 'logprobs': None} id='lc_run--019ce7ce-0b50-7051-a3c8-0b8d32b027ea-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 51, 'output_tokens': 14, 'total_tokens': 65, 'input_token_details': {}, 'output_token_details': {}}


In [20]:
print(custom_response.content)

I'm heading to the store to pick up a few groceries.


In [21]:
{
    "gift": False,
    "delivery_days": 5,
    "price_value": "pretty affordable!"
}

{'gift': False, 'delivery_days': 5, 'price_value': 'pretty affordable!'}

In [22]:
custom_review = """
This leaf blower is pretty amazing.It has four settings:\
candle blower, gentle breeze, windy city, and tornado. \
It arrived in two days, just in time for my wife's \
anniversary present.\
I think my wife liked it so much she was speechless. \
So far I've been the only one using it, and I've been \
using it every other morning to clear the leaves on our lawrIt's slightly more expensive than the other leaf blowers \
out there, but I think it's worth it for the extra features
"""

review_template = """
For the following text, extract the following information:
gift: Was the item purchased as a gift for someone else? Ansdelivery days: How many days did it take for the product toprice_value: Extract any sentences about the value or price,
Format the output as JSON with the following keys:
gift
delivery_days
price_value

text: {text}
"""


In [23]:
from langchain_core.prompts import ChatPromptTemplate
prompt_template = ChatPromptTemplate.from_template(review_template)
print(prompt_template)

input_variables=['text'] input_types={} partial_variables={} messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['text'], input_types={}, partial_variables={}, template='\nFor the following text, extract the following information:\ngift: Was the item purchased as a gift for someone else? Ansdelivery days: How many days did it take for the product toprice_value: Extract any sentences about the value or price,\nFormat the output as JSON with the following keys:\ngift\ndelivery_days\nprice_value\n\ntext: {text}\n'), additional_kwargs={})]


In [24]:
message = prompt_template.format_messages(text=custom_review)
chat = ChatOpenAI(api_key=os.getenv("OPENAI_API_KEY"),
                  base_url="https://api.longcat.chat/openai",
                  model="LongCat-Flash-Chat",
                  max_tokens=1000,
                  temperature=0)
response = chat.invoke(message)
print(response)
print(response.content)


content='{\n  "gift": true,\n  "delivery_days": 2,\n  "price_value": "It\'s slightly more expensive than the other leaf blowers out there, but I think it\'s worth it for the extra features"\n}' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 50, 'prompt_tokens': 193, 'total_tokens': 243, 'completion_tokens_details': None, 'prompt_tokens_details': None, 'cache_write_tokens': 0, 'cache_read_tokens': 0, 'input_tokens': 0, 'output_tokens': 0, 'cached_tokens': 0}, 'model_provider': 'openai', 'model_name': 'longcat-flash-chatai-api', 'system_fingerprint': None, 'id': 'c3db8616792947d5ade23854d7adedaf', 'finish_reason': 'stop', 'logprobs': None} id='lc_run--019ce7ce-10ee-7a81-8873-c7908c73b63c-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 193, 'output_tokens': 50, 'total_tokens': 243, 'input_token_details': {}, 'output_token_details': {}}
{
  "gift": true,
  "delivery_days": 2,
  "price_value": "It's slightly more expensive 

In [25]:
type(response.content)

str

这里返回的是字符串而不是json格式，直接调用get会报错

In [26]:
response.content.get("gift")

AttributeError: 'str' object has no attribute 'get'

调用Langchain的解析方法解析答案

In [27]:
from langchain_core.output_parsers import JsonOutputParser
from pydantic import BaseModel, Field  # 直接用 pydantic 官方包

In [28]:
# 用 Pydantic 模型替代 ResponseSchema 列表
class ProductInfo(BaseModel):
    gift: str = Field(description="Was the item purchased as a gift for someone else?")  # 补全你原来的 description
    delivery_days: str = Field(description="How many days did it take for the product to arrive?")  # 补全你原来的 description
    price_value: str = Field(description="Extract any sentences about the value or price")  # 补全你原来的 description


# 初始化解析器
parser = JsonOutputParser(pydantic_object=ProductInfo)

gift_schema = parser.get_format_instructions()
delivery_days_schema = parser.get_format_instructions()
price_value_schema = parser.get_format_instructions()
response_schemas = [gift_schema, delivery_days_schema, price_value_schema]

print(gift_schema)

STRICT OUTPUT FORMAT:
- Return only the JSON value that conforms to the schema. Do not include any additional text, explanations, headings, or separators.
- Do not wrap the JSON in Markdown or code fences (no ``` or ```json).
- Do not prepend or append any text (e.g., do not write "Here is the JSON:").
- The response must be a single top-level JSON value exactly as required by the schema (object/array/etc.), with no trailing commas or comments.

The output should be formatted as a JSON instance that conforms to the JSON schema below.

As an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]} the object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.

Here is the output schema (shown in a code block for readability only — do not include any backticks or Markdown in your output):


In [29]:
format_instructions = parser.get_format_instructions()
print(format_instructions)

STRICT OUTPUT FORMAT:
- Return only the JSON value that conforms to the schema. Do not include any additional text, explanations, headings, or separators.
- Do not wrap the JSON in Markdown or code fences (no ``` or ```json).
- Do not prepend or append any text (e.g., do not write "Here is the JSON:").
- The response must be a single top-level JSON value exactly as required by the schema (object/array/etc.), with no trailing commas or comments.

The output should be formatted as a JSON instance that conforms to the JSON schema below.

As an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]} the object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.

Here is the output schema (shown in a code block for readability only — do not include any backticks or Markdown in your output):


In [30]:
review_template = """
For the following text, extract the following information:
gift: Was the item purchased as a gift for someone else? 
delivery days: How many days did it take for the product to
price_value: Extract any sentences about the value or price

text: {text}

{format_instructions}
"""

In [31]:
prompt_template = ChatPromptTemplate.from_template(review_template)
message = prompt_template.format_messages(text=custom_review, format_instructions=format_instructions)

In [32]:
chat = ChatOpenAI(api_key=os.getenv("OPENAI_API_KEY"),
                  base_url="https://api.longcat.chat/openai",
                  model="LongCat-Flash-Chat",
                  max_tokens=1000,
                  temperature=0)

In [33]:
response = chat.invoke(prompt_template.format_messages(text=custom_review, format_instructions=format_instructions))

In [34]:
print(response.content)

{"gift": "yes", "delivery_days": "2", "price_value": "It's slightly more expensive than the other leaf blowers out there, but I think it's worth it for the extra features"}


In [35]:
type(response.content)

str

In [36]:
parsed_output = parser.parse(response.content)

In [37]:
type(parsed_output)

dict

In [38]:
print(parsed_output.get("gift"))

yes


这里应该修改prompt返回bool值的，懒得改了